In [ ]:
!pip install scanpy seaborn igraph leidenalg

### Imports

We import scanpy as the main analysis library and seaborn for plotting
QC histograms. `set_figure_params` sets a white background for all plots,
and `verbosity = 3` makes scanpy print progress messages during computation.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc
import seaborn as sns

sc.set_figure_params(facecolor="white", figsize=(8, 8))
sc.settings.verbosity = 3

### Loading the Data

`visium_sge` downloads the human lymph node Visium dataset from 10x Genomics
and loads it as an AnnData object. AnnData stores the count matrix in `.X`,
spatial coordinates in `.obsm["spatial"]`, and the tissue image in `.uns["spatial"]`.

We flag mitochondrial genes (those starting with "MT-") since a high percentage
of mitochondrial reads usually indicates a low-quality or dying cell.
`calculate_qc_metrics` then computes per-spot statistics like total counts,
number of genes detected, and the percentage of mitochondrial counts.

In [ ]:
adata = sc.datasets.visium_sge(sample_id="V1_Human_Lymph_Node")
adata.var_names_make_unique()
adata.var["mt"] = adata.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], inplace=True)

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(15, 4))
sns.histplot(adata.obs["total_counts"], kde=False, ax=axs[0])
sns.histplot(adata.obs["total_counts"][adata.obs["total_counts"] < 10000], kde=False, bins=40, ax=axs[1])
sns.histplot(adata.obs["n_genes_by_counts"], kde=False, bins=60, ax=axs[2])
sns.histplot(adata.obs["n_genes_by_counts"][adata.obs["n_genes_by_counts"] < 4000], kde=False, bins=60, ax=axs[3])
plt.savefig("qc_histograms.png")

### QC Plots

We plot the distribution of total counts and detected genes per spot.
The first two panels show total counts at full range and zoomed in below
10,000, which helps us decide a sensible upper cutoff. The last two do
the same for number of genes detected. Spots at the extremes (very low
or very high counts) are likely of poor quality or technical artifacts.

**Note:** We added `plt.savefig("qc_histograms.png")` to save this figure.

In [ ]:
sc.pp.filter_cells(adata, min_counts=5000)
sc.pp.filter_cells(adata, max_counts=35000)
adata = adata[adata.obs["pct_counts_mt"] < 20].copy()
print(f"#cells after MT filter: {adata.n_obs}")
sc.pp.filter_genes(adata, min_cells=10)

### Filtering

We remove spots with fewer than 5,000 or more than 35,000 total counts,
and spots where more than 20% of counts come from mitochondrial genes.
We also remove genes detected in fewer than 10 spots, since genes expressed
in almost no spots carry little useful information and just add noise.

In [ ]:
sc.pp.normalize_total(adata, inplace=True)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, flavor="seurat", n_top_genes=2000)

### Normalization

`normalize_total` scales each spot so that all spots have the same total
count, removing technical differences in sequencing depth. `log1p` then
applies a log(x+1) transformation to stabilize variance — highly expressed
genes would otherwise dominate downstream analysis.

`highly_variable_genes` selects the 2,000 genes that vary the most across
spots. These are the genes most likely to carry biological signal, so
subsequent steps like PCA are run on this subset.

In [ ]:
sc.pp.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.tl.leiden(adata, key_added="clusters", flavor="igraph", directed=False, n_iterations=2)

### Dimensionality Reduction and Clustering

PCA compresses the ~2,000 gene dimensions into 50 principal components,
capturing most of the variance. `neighbors` then builds a k-nearest-neighbor
graph in PCA space — spots with similar gene expression profiles are
connected. UMAP uses this graph to produce a 2D embedding for visualization.

Leiden clustering runs on the same neighbor graph and groups spots into
communities based on transcriptional similarity. The result is stored in
`adata.obs["clusters"]`.

In [ ]:
plt.rcParams["figure.figsize"] = (4, 4)
sc.pl.umap(adata, color=["total_counts", "n_genes_by_counts", "clusters"], wspace=0.4, show=False)
plt.savefig("umap_plots.png")

### UMAP Visualization

We color the UMAP by total counts, number of genes, and cluster label.
The first two help confirm that the clusters are not simply driven by
sequencing depth. If count-rich spots all clump together, that's a sign
of a technical artifact rather than true biology.

**Note:** We added `plt.savefig("umap_plots.png")` to export this visualization.

In [ ]:
plt.rcParams["figure.figsize"] = (8, 8)
sc.pl.spatial(adata, img_key="hires", color=["total_counts", "n_genes_by_counts"], show=False)
plt.savefig("spatial_counts.png")
sc.pl.spatial(adata, img_key="hires", color="clusters", size=1.5, show=False)
plt.savefig("spatial_clusters.png")

### Spatial Visualization

`sc.pl.spatial` overlays colored circles on top of the H&E tissue image,
one circle per Visium spot. Here we check whether spots with high counts
or many detected genes appear in particular tissue regions, and then
visualize how the transcriptional clusters map onto the tissue.

Spots from the same cluster often co-localize in the tissue, which suggests
the clusters correspond to real tissue structures or cell populations.

**Note:** We added `plt.savefig` calls to generate `spatial_counts.png` and `spatial_clusters.png`.

In [ ]:
sc.pl.spatial(adata, img_key="hires", color="clusters",
              groups=["5", "9"], crop_coord=[7000, 10000, 0, 6000], alpha=0.5, size=1.3, show=False)
plt.savefig("spatial_zoomed_clusters.png")

### Identifying Highly Variable Genes

Highly variable genes (HVGs) are genes that show significant variation across the cells in your dataset. Selecting these genes helps focus downstream analysis on the most biologically informative features.

In Scanpy, we use `sc.pp.highly_variable_genes`. The `flavor="seurat"` parameter is common for count data, and `n_top_genes` allows us to specify exactly how many genes we want to keep for the next steps.

**Note:** We added `plt.savefig("hvg_plot.png")` to export this visualization.

In [ ]:
# Identify 2,000 highly variable genes
sc.pp.highly_variable_genes(adata, flavor="seurat", n_top_genes=2000)

# Visualize the dispersion vs mean to see the selection
sc.pl.highly_variable_genes(adata, show=False)
plt.savefig("hvg_plot.png")

# We can check how many genes are flagged as highly variable
print(f"Number of HVGs: {adata.var['highly_variable'].sum()}")

### Zooming In

We crop to a specific region of the tissue and reduce spot opacity to 0.5
so the underlying H&E morphology shows through. This lets us visually
inspect which tissue structures correspond to which clusters.

**Note:** This zoomed view is now saved as `spatial_zoomed_clusters.png`.

### Marker Genes

`rank_genes_groups` runs a t-test for each cluster versus all other clusters
to find genes that are differentially expressed in that cluster. We visualize
the top 10 markers for cluster 9 as a heatmap across all clusters.

We then plot the spatial expression of CR2 (a top marker) alongside the
cluster map to confirm that the gene's expression pattern matches the
cluster's spatial distribution. COL1A2 and SYPL1 are also plotted as
further examples.

**Note:** We added `plt.savefig` to each step to output `rank_genes_heatmap.png`, `spatial_cr2_marker.png`, and `spatial_markers_additional.png`.

In [ ]:
sc.tl.rank_genes_groups(adata, "clusters", method="t-test")
sc.pl.rank_genes_groups_heatmap(adata, groups="9", n_genes=10, groupby="clusters", show=False)
plt.savefig("rank_genes_heatmap.png")
sc.pl.spatial(adata, img_key="hires", color=["clusters", "CR2"], show=False)
plt.savefig("spatial_cr2_marker.png")
sc.pl.spatial(adata, img_key="hires", color=["COL1A2", "SYPL1"], alpha=0.7, show=False)
plt.savefig("spatial_markers_additional.png")